In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
customers_dim_base = spark.table("retail_silver.silver1_customers_clean")

display(customers_dim_base.limit(5))

customer_id,customer_name,city,segment,gender,signup_date,status,source_file,ingestion_ts,load_type
C00001,Customer_1,Delhi,Regular,Other,2024-10-08,active,customers_batch.csv,2026-07-09T19:08:51.698Z,batch
C00002,Customer_2,Bengaluru,Silver,Other,2024-04-14,active,customers_batch.csv,2026-07-09T19:08:51.698Z,batch
C00003,Customer_3,Gurugram,Platinum,M,2024-01-31,active,customers_batch.csv,2026-07-09T19:08:51.698Z,batch
C00004,Customer_4,Bengaluru,Silver,Other,2025-09-08,active,customers_batch.csv,2026-07-09T19:08:51.698Z,batch
C00005,Customer_5,Ahmedabad,Silver,Other,2025-10-27,inactive,customers_batch.csv,2026-07-09T19:08:51.698Z,batch


In [0]:
customers_dim_base.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- load_type: string (nullable = true)



In [0]:
customer_scd2_initial = (
    customers_dim_base
    .withColumn("customer_sk", monotonically_increasing_id())
    .withColumn("effective_start_date", col("signup_date"))
    .withColumn("effective_end_date", lit("9999-12-31").cast("date"))
    .withColumn("is_current", lit(True))
    .withColumn(
        "hash_value",
        sha2(concat_ws("||", "customer_name", "city", "segment", "gender", "status"), 256)
    )
)

In [0]:
customer_scd2_initial.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_silver.dim_customer_scd2")

In [0]:
spark.sql("""
SELECT COUNT(*) AS total_customer_scd2
FROM retail_silver.dim_customer_scd2
""").show()

+-------------------+
|total_customer_scd2|
+-------------------+
|               2535|
+-------------------+



In [0]:
customers_cdc = spark.table("retail_raw.bronze_customers_cdc")

display(customers_cdc.limit(5))

customer_id,customer_name,city,segment,gender,signup_date,status,effective_date,operation,source_file,ingestion_ts,load_type
C00938,Customer_938,Mumbai,Platinum,Other,2025-08-02,active,2026-04-24,UPDATE,customers_cdc,2026-07-09T19:20:25.515Z,incremental
C01955,Customer_1955,Hyderabad,Silver,F,2025-03-14,inactive,2026-04-24,UPDATE,customers_cdc,2026-07-09T19:20:25.515Z,incremental
C01539,Customer_1539,Ahmedabad,Silver,M,2024-09-08,active,2026-04-24,UPDATE,customers_cdc,2026-07-09T19:20:25.515Z,incremental
C00577,Customer_577,Delhi,Gold,F,2025-11-17,inactive,2026-04-24,UPDATE,customers_cdc,2026-07-09T19:20:25.515Z,incremental
C01999,Customer_1999,Ahmedabad,Silver,M,2024-11-29,active,2026-04-24,UPDATE,customers_cdc,2026-07-09T19:20:25.515Z,incremental


In [0]:
customers_cdc.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- effective_date: string (nullable = true)
 |-- operation: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- load_type: string (nullable = true)



In [0]:
customers_cdc_clean = (
    customers_cdc
    .filter(col("customer_id").isNotNull())
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("city", coalesce(trim(col("city")), lit("Unknown")))
    .withColumn("segment", coalesce(trim(col("segment")), lit("Unknown")))
    .withColumn("gender", coalesce(trim(col("gender")), lit("Unknown")))
    .withColumn("signup_date", expr("try_cast(signup_date as date)"))
    .withColumn("effective_date", expr("try_cast(effective_date as date)"))
    .withColumn("status", coalesce(trim(col("status")), lit("Unknown")))
    .withColumn(
        "hash_value",
        sha2(concat_ws("||", "customer_name", "city", "segment", "gender", "status"), 256)
    )
)

In [0]:
customers_cdc_valid = customers_cdc_clean.filter(
    col("effective_date").isNotNull()
)

In [0]:
print("Customer CDC Raw:", customers_cdc.count())
print("Customer CDC Valid:", customers_cdc_valid.count())

Customer CDC Raw: 630
Customer CDC Valid: 629


In [0]:
current_customers = spark.table("retail_silver.dim_customer_scd2").filter(col("is_current") == True)

changed_customers = (
    customers_cdc_valid.alias("src")
    .join(
        current_customers.alias("tgt"),
        col("src.customer_id") == col("tgt.customer_id"),
        "inner"
    )
    .filter(col("src.hash_value") != col("tgt.hash_value"))
    .select("src.*")
)

print("Changed Customers:", changed_customers.count())

Changed Customers: 0


In [0]:
window_cdc_customer = Window.partitionBy("customer_id").orderBy(col("effective_date").desc(), col("ingestion_ts").desc())

changed_customers_dedup = (
    changed_customers
    .withColumn("rn", row_number().over(window_cdc_customer))
    .filter(col("rn") == 1)
    .drop("rn")
)

print("Changed Customers Before Dedup:", changed_customers.count())
print("Changed Customers After Dedup:", changed_customers_dedup.count())

Changed Customers Before Dedup: 0
Changed Customers After Dedup: 0


In [0]:
customer_delta = DeltaTable.forName(spark, "retail_silver.dim_customer_scd2")

customer_delta.alias("tgt").merge(
    changed_customers_dedup.alias("src"),
    "tgt.customer_id = src.customer_id AND tgt.is_current = true"
).whenMatchedUpdate(
    set={
        "effective_end_date": "date_sub(src.effective_date, 1)",
        "is_current": "false"
    }
).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
SELECT is_current, COUNT(*) AS count
FROM retail_silver.dim_customer_scd2
GROUP BY is_current
""").show()

+----------+-----+
|is_current|count|
+----------+-----+
|      true| 2187|
|     false|  348|
+----------+-----+



In [0]:
updated_customer_rows = (
    customers_cdc_valid
    .withColumn("customer_sk", monotonically_increasing_id())
    .withColumn("effective_start_date", col("effective_date"))
    .withColumn("effective_end_date", lit("9999-12-31").cast("date"))
    .withColumn("is_current", lit(True))
    .select(
        "customer_sk",
        "customer_id",
        "customer_name",
        "city",
        "segment",
        "gender",
        "signup_date",
        "status",
        "source_file",
        "ingestion_ts",
        "load_type",
        "effective_start_date",
        "effective_end_date",
        "is_current",
        "hash_value"
    )
)

print("Rows to insert:", updated_customer_rows.count())



Rows to insert: 629


In [0]:
updated_customer_rows.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("retail_silver.dim_customer_scd2")

In [0]:
spark.sql("""
SELECT is_current, COUNT(*) AS count
FROM retail_silver.dim_customer_scd2
GROUP BY is_current
""").show()

+----------+-----+
|is_current|count|
+----------+-----+
|      true| 2816|
|     false|  348|
+----------+-----+



In [0]:
spark.sql("""
SELECT COUNT(*) AS total_customer_scd2
FROM retail_silver.dim_customer_scd2
""").show()

+-------------------+
|total_customer_scd2|
+-------------------+
|               3164|
+-------------------+



In [0]:
products_dim_base = spark.table("retail_silver.silver1_products_clean")

display(products_dim_base.limit(5))

product_id,product_name,category,brand,unit_price,status,created_date,source_file,ingestion_ts,load_type
P00003,Bedsheet 3,Home,BrandD,30753.26,active,2024-10-23,products_batch.csv,2026-07-09T19:08:58.563Z,batch
P00004,Oil 4,Grocery,BrandB,68176.44,active,2024-03-07,products_batch.csv,2026-07-09T19:08:58.563Z,batch
P00005,Oil 5,Unknown,BrandC,51796.39,active,2024-12-12,products_batch.csv,2026-07-09T19:08:58.563Z,batch
P00006,Chair 6,Home,BrandC,10740.11,discontinued,2025-06-17,products_batch.csv,2026-07-09T19:08:58.563Z,batch
P00007,Jeans 7,Fashion,BrandC,26020.02,active,2024-04-19,products_batch.csv,2026-07-09T19:08:58.563Z,batch


In [0]:
product_scd2_initial = (
    products_dim_base
    .withColumn("product_sk", monotonically_increasing_id())
    .withColumn("effective_start_date", col("created_date"))
    .withColumn("effective_end_date", lit("9999-12-31").cast("date"))
    .withColumn("is_current", lit(True))
    .withColumn(
        "hash_value",
        sha2(concat_ws("||", "product_name", "category", "brand", "unit_price", "status"), 256)
    )
)

In [0]:
product_scd2_initial.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_silver.dim_product_scd2")

In [0]:
spark.sql("""
SELECT COUNT(*) AS total_product_scd2
FROM retail_silver.dim_product_scd2
""").show()

+------------------+
|total_product_scd2|
+------------------+
|               810|
+------------------+



In [0]:
products_cdc = spark.table("retail_raw.bronze_products_cdc")

display(products_cdc.limit(5))

product_id,product_name,category,brand,unit_price,status,created_date,effective_date,operation,source_file,ingestion_ts,load_type
P00435,Bedsheet 435,Home,BrandB,34748.71,active,2023-12-16,2026-04-24,UPDATE,products_cdc,2026-07-09T19:21:41.638Z,incremental
P00140,Mouse 140,Electronics,BrandB,13985.55,active,2025-02-21,2026-04-24,UPDATE,products_cdc,2026-07-09T19:21:41.638Z,incremental
P00081,Shampoo 81,Beauty,BrandD,73001.3,discontinued,2025-10-01,2026-04-24,UPDATE,products_cdc,2026-07-09T19:21:41.638Z,incremental
P00561,Chair 561,Home,BrandC,45448.0,active,2025-10-14,2026-04-24,UPDATE,products_cdc,2026-07-09T19:21:41.638Z,incremental
P00187,Rice 187,Grocery,BrandD,9702.61,active,2025-09-18,2026-04-24,UPDATE,products_cdc,2026-07-09T19:21:41.638Z,incremental


In [0]:
products_cdc.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- status: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- effective_date: string (nullable = true)
 |-- operation: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- load_type: string (nullable = true)



In [0]:
products_cdc_clean = (
    products_cdc
    .filter(col("product_id").isNotNull())
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", coalesce(trim(col("category")), lit("Unknown")))
    .withColumn("brand", coalesce(trim(col("brand")), lit("Unknown")))
    .withColumn("unit_price", expr("try_cast(regexp_replace(unit_price,'[^0-9.]','') as double)"))
    .withColumn("status", coalesce(trim(col("status")), lit("Unknown")))
    .withColumn("created_date", expr("try_cast(created_date as date)"))
    .withColumn("effective_date", expr("try_cast(effective_date as date)"))
    .withColumn(
        "hash_value",
        sha2(concat_ws("||", "product_name", "category", "brand", "unit_price", "status"), 256)
    )
)

products_cdc_valid = products_cdc_clean.filter(
    col("effective_date").isNotNull() &
    col("unit_price").isNotNull()
)

print("Product CDC Raw:", products_cdc.count())
print("Product CDC Valid:", products_cdc_valid.count())

Product CDC Raw: 180
Product CDC Valid: 180


In [0]:
current_products = spark.table("retail_silver.dim_product_scd2").filter(
    col("is_current") == True
)

changed_products = (
    products_cdc_valid.alias("src")
    .join(
        current_products.alias("tgt"),
        col("src.product_id") == col("tgt.product_id"),
        "inner"
    )
    .filter(col("src.hash_value") != col("tgt.hash_value"))
    .select("src.*")
)

print("Changed Products:", changed_products.count())

Changed Products: 181


In [0]:
window_cdc_product = Window.partitionBy("product_id").orderBy(
    col("effective_date").desc(),
    col("ingestion_ts").desc()
)

changed_products_dedup = (
    changed_products
    .withColumn("rn", row_number().over(window_cdc_product))
    .filter(col("rn") == 1)
    .drop("rn")
)

print("Changed Products Before Dedup:", changed_products.count())
print("Changed Products After Dedup:", changed_products_dedup.count())

Changed Products Before Dedup: 181
Changed Products After Dedup: 162


In [0]:
product_delta = DeltaTable.forName(spark, "retail_silver.dim_product_scd2")

product_delta.alias("tgt").merge(
    changed_products_dedup.alias("src"),
    "tgt.product_id = src.product_id AND tgt.is_current = true"
).whenMatchedUpdate(
    set={
        "effective_end_date": "date_sub(src.effective_date, 1)",
        "is_current": "false"
    }
).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
SELECT is_current, COUNT(*) AS count
FROM retail_silver.dim_product_scd2
GROUP BY is_current
""").show()

+----------+-----+
|is_current|count|
+----------+-----+
|      true|  642|
|     false|  168|
+----------+-----+



In [0]:
updated_product_rows = (
    products_cdc_valid
    .withColumn("product_sk", monotonically_increasing_id())
    .withColumn("effective_start_date", col("effective_date"))
    .withColumn("effective_end_date", lit("9999-12-31").cast("date"))
    .withColumn("is_current", lit(True))
    .select(
        "product_sk",
        "product_id",
        "product_name",
        "category",
        "brand",
        "unit_price",
        "status",
        "created_date",
        "source_file",
        "ingestion_ts",
        "load_type",
        "effective_start_date",
        "effective_end_date",
        "is_current",
        "hash_value"
    )
)

print("Rows to insert:", updated_product_rows.count())

Rows to insert: 180


In [0]:
updated_product_rows.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("retail_silver.dim_product_scd2")

In [0]:
spark.sql("""
SELECT is_current, COUNT(*) AS count
FROM retail_silver.dim_product_scd2
GROUP BY is_current
""").show()

+----------+-----+
|is_current|count|
+----------+-----+
|      true|  822|
|     false|  168|
+----------+-----+



In [0]:
spark.sql("""
SELECT 'dim_customer_scd2' AS table_name, COUNT(*) AS row_count
FROM retail_silver.dim_customer_scd2

UNION ALL

SELECT 'dim_product_scd2', COUNT(*)
FROM retail_silver.dim_product_scd2
""").show()

+-----------------+---------+
|       table_name|row_count|
+-----------------+---------+
|dim_customer_scd2|     3164|
| dim_product_scd2|      990|
+-----------------+---------+

